# TP 3 — Stabiliser puis transformer chaque token

Le but de cette partie c'est de comprendre comment marche la LayerNorm (pour stabiliser les valeurs) et le Feed-Forward (le petit réseau de neurones qu'on applique sur chaque token individuellement).

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 : coder LayerNorm

In [2]:
class MaLayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        moyenne = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - moyenne) / torch.sqrt(variance + self.eps)
        return self.gamma * x_norm + self.beta

**Questions d'interprétation** — Sur quel axe normaliser ? À quoi servent ε, γ et β ?

**Réponses :**
- On fait la normalisation sur la toute dernière dimension (dim=-1), c'est-à-dire sur les features du token. Chaque token est recentré par rapport à lui-même.
- epsilon sert juste à éviter qu'on divise par zéro si la variance est nulle.
- gamma et beta sont des paramètres que le réseau va pouvoir apprendre. Si la normalisation casse une info importante, le réseau peut s'en servir pour annuler l'effet en modifiant l'échelle ou en décalant les valeurs.

## Étape 2 : choisir l'axe de normalisation

In [3]:
torch.manual_seed(1337)
mon_module_norm = MaLayerNorm(100)

entrees = torch.randn(32, 100)
entrees_norm = mon_module_norm(entrees)

print("Moyenne/std sur la colonne 0:", entrees_norm[:, 0].mean().item(), entrees_norm[:, 0].std().item())
print("Moyenne/std sur la ligne 0:", entrees_norm[0, :].mean().item(), entrees_norm[0, :].std().item())

Moyenne/std sur la colonne 0: 0.14759676158428192 0.8847486972808838
Moyenne/std sur la ligne 0: -2.2649764730431343e-08 1.005033254623413


**Questions d'interprétation** — Prédisez quelle ligne donnera environ (0, 1) avant d'exécuter le code.

Validez enfin le résultat avec `torch.allclose`.

**Réponses :**
- C'est la ligne 0 qui va donner une moyenne de 0 et une variance de 1. Vu qu'on normalise token par token, c'est logique que l'ensemble des 100 features d'un seul token (la ligne) soit normalisé. Les colonnes, elles, ne sont pas contraintes.
- On peut tester avec `torch.allclose(mon_module_norm(entrees), nn.LayerNorm(100)(entrees), atol=1e-6)` et ça renvoie bien True, notre code fait la même chose que la version officielle de PyTorch.

## Étape 3 : construire le réseau feed-forward

In [4]:
class ReseauFeedForward(nn.Module):
    def __init__(self, dim_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_emb, 4 * dim_emb),
            nn.ReLU(),
            nn.Linear(4 * dim_emb, dim_emb),
        )

    def forward(self, x):
        return self.net(x)

mon_ff = ReseauFeedForward(100)
delta_test = mon_ff(torch.randn(32, 100))
print("Forme à la sortie :", delta_test.shape)

Forme à la sortie : torch.Size([32, 100])


**Questions d'interprétation** — Quelle proportion d'activations est positive ? Pourquoi revenir à C ?

**Réponses :**
- Y a à peu près 50% des activations qui sont positives. Comme on a initialisé avec du random centré sur 0, le ReLU va juste couper tout ce qui est négatif (soit environ la moitié).
- On est obligés de revenir à la taille C à la fin pour pouvoir faire l'addition résiduelle (`x = x + output`). Le passage par 4*C au milieu sert juste à augmenter temporairement le nombre de dimensions pour que le réseau puisse faire des calculs plus complexes.

## Étapes 4 et 5 : ajouter le résiduel

On teste l'organisation pré-norm.

In [5]:
B, T, d_emb = 4, 8, 32

norm = nn.LayerNorm(d_emb)
ff = ReseauFeedForward(d_emb)

x_entree = torch.randn(B, T, d_emb)
delta_ff = ff(norm(x_entree))
x_final = x_entree + delta_ff

print(x_entree.shape)
print(delta_ff.shape)
print(x_final.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


**Bilan du TP 3** — Quelle dimension LayerNorm conserve-t-elle ? Le MLP mélange-t-il les tokens ? À quoi sert la connexion résiduelle ?

**Bilan des tests et réponses :**
- Si on testait GELU à la place de ReLU, le comportement de la couche changerait légèrement (GELU laisse filtrer un tout petit peu les valeurs négatives, ce que font beaucoup les modèles récents comme GPT) mais ça ne casse pas les dimensions.
- Concernant la LayerNorm : elle garde toutes les dimensions intactes (B, T, C). Elle se contente de modifier les valeurs à l'intérieur.
- Le Feed-Forward (MLP) ne mélange absolument pas les tokens. Il applique exactement la même opération mathématique à chaque token indépendamment des autres.
- La connexion résiduelle (`x = x + delta`) est vitale : elle permet au gradient de circuler super facilement quand on fait la backpropagation. Sans ça (si on faisait juste `x_out = delta`), les réseaux profonds auraient énormément de mal à apprendre à cause du problème de disparition du gradient.